**Implementing the Demirel et al. paper**

In [56]:
import numpy as np
from scipy import sparse
from pygsp import graphs
import math
import matplotlib.pyplot as plt
from typing import NamedTuple


**Initialisation of factors and building factor graph**

In [ ]:
class Factor(NamedTuple):
    vars: list[int]     # the node indices this factor touches
    C: list[float]      # coefficient on each, aligned with vars
    z: float            # measurement
    var_z: float        # measurement variance

def build_factor_graph(y, sigma, edges, gamma):
    N = len(y) # placeholder
    factors = []
    for i in range(N):
        factors.append(Factor([i], [1], y[i], sigma[i]))

    for i, j, w in edges:
        factors.append(Factor([i, j], [1, -1], 0, 1/(gamma*w)))

    eps = 1e-4
    for i in range(N):
        factors.append(Factor([i], [1.0], 0.0, 1/eps))

    var_factors = [[] for _ in range(N)]
    for f_id, f in enumerate(factors):
        for s in f.vars:
            var_factors[s].append(f_id)
    
    return factors, var_factors

"""
# edge table from worked example with weights edge = (node1, node2, weight)
edges = [
    (0, 1, 1.5),
    (1, 2, 2),
    (2, 3, 1.8),
    (3, 4, 1.2),
    (1, 4, 0.8),
    (4, 5, 2.1),
    (5, 6, 1.6)
]
"""

sigma = np.array([0.25, 0.25, 0.25, 225, 0.25, 0.25, 0.25])             # drift injection - variances  with additional drift at node 4.
y = np.array([120.15, 114.48, 108.38, 108.47, 109.02, 94.35, 85.06])    # measurements with drift injection - N(0, 0.25)
gamma = 0.3

(factors, var_factors) = build_factor_graph(y, sigma, edges, gamma)

"""Testing
N = len(y)
Lambda = np.zeros((N,N))
b = np.zeros(N)
for f in factors:
    c = np.zeros(N)
    c[f.vars] = f.C
    Lambda += np.outer(c, c) / f.var_z
    b += c * f.z / f.var_z
np.linalg.solve(Lambda, b)
"""


'Testing\nN = len(y)\nLambda = np.zeros((N,N))\nb = np.zeros(N)\nfor f in factors:\n    c = np.zeros(N)\n    c[f.vars] = f.C\n    Lambda += np.outer(c, c) / f.var_z\n    b += c * f.z / f.var_z\nnp.linalg.solve(Lambda, b)\n'

**GaBP Algorithm**

In [ ]:
"""
factors     : the list of factors from NamedTuple class - Factor(vars, C, z, var_z)
var_factors : var_factors[s] = the list of factors with id (f_id) touching variable s
v2f, f2v    : variable-to-factor & factor-to-variable respectively
"""

def links(factors):
    """Function that produces a lazy generator object as and when we need it in loops"""
    for f_id, f in enumerate(factors):
        for s in f.vars:
            yield (f_id, s)

def init_messages(factors):
    """Initilaises uninformative prior for messages i.e (Lam, eta) = (0, 0)"""
    return {(f_id, s): (0.0, 0.0) for f_id, s in links(factors)}

def var_to_factor(f_id, s, f2v, var_factors):
    """Variable to factor message: just a sum over the other factors on s"""
    Lam = eta = 0.0
    for fp in var_factors[s]:
        if fp == f_id: # (excludes the factor we are sending the message to)
            continue
        Lf, ef = f2v[(fp, s)] # (f -> v incoming messages)
        Lam += Lf
        eta += ef
    return (Lam, eta)

def factor_to_var(f_id, s, v2f, factors):
    f = factors[f_id]
    Cs = f.C[f.vars.index(s)]
    acc_mean = acc_var = 0.0
    for b, Cb in zip(f.vars, f.C):
        if b == s:
            continue
        Lb, eb = v2f[(f_id, b)] # (v -> f incoming messages)
        if Lb == 0.0:
            return (0.0, 0.0)
        mu_b, var_b = eb / Lb, 1.0 / Lb
        acc_mean += Cb * mu_b
        acc_var += Cb **2 * var_b
    mean = (f.z - acc_mean) / Cs
    var = (f.var_z + acc_var) / Cs**2
    Lam = 1.0 / var
    return (Lam, mean*Lam)

def belief(s, f2v, var_factors):
    """Our belief at s which is just a sum over factors on s"""
    Lam = eta = 0.0
    for fp in var_factors[s]:
        Lf, ef = f2v[(fp,s)]
        Lam += Lf
        eta += ef
    return (eta / Lam, 1.0 / Lam) # (just (mean, variance))

def run_gabp(factors, var_factors, N, max_iters=20, tol=1e-8):
    """Synchronous flooding"""
    v2f = init_messages(factors)
    f2v = init_messages(factors)
    prev = None
    for _ in range(max_iters):
        v2f = {(f_id, s): var_to_factor(f_id, s, f2v, var_factors) for f_id, s in links(factors)}
        f2v = {(f_id, s): factor_to_var(f_id, s, v2f, factors) for f_id, s in links(factors)}
        cur = {s: belief(s, f2v, var_factors) for s in range(N)}

        def converged(cur, prev, tol):
            deltas = []
            for s in cur:
                mean, var = cur[s]
                mean_prev, var_prev = prev[s]
                deltas.append(abs(mean - mean_prev))
                deltas.append(abs(var - var_prev))
            return max(deltas) < tol

        if prev is not None and converged(cur, prev, tol):
            break
        prev = cur
    return cur

cur = run_gabp(factors, var_factors, N, max_iters=20, tol=1e-8)
print(cur)
print(len(factors))


{0: (np.float64(119.52282188853557), np.float64(0.2266964314345328)), 1: (np.float64(113.97446596900421), np.float64(0.19386590406556337)), 2: (np.float64(109.03572369094051), np.float64(0.21098418567392768)), 3: (np.float64(108.42563659436063), np.float64(1.216792351002852)), 4: (np.float64(107.54007648501481), np.float64(0.2005014017234595)), 5: (np.float64(95.2035762191732), np.float64(0.20079006653671153)), 6: (np.float64(86.1448888609588), np.float64(0.22551418847467133))}
21


'\n[119.52282189 113.97446597 109.03572369 108.42563659 107.54007649 95.20357622  86.14488886] \n[0.22669742 0.19396252 0.21108934 1.21739878 0.20060133 0.20079162 0.22551421]\n'

**Darcy-Weisbach equations implementation**

In [ ]:
# (i, j, R_ij)
gas_edges = [
    (0, 1, 0.5),
    (1, 2, 1.2),
    (2, 3, 0.8),
    (3, 4, 2.0),
    (1, 4, 1.5),
    (4, 5, 0.6),
    (5, 6, 1.0),
]

neighbours = {i: [] for i in range(N)}
for i, j, R in gas_edges:
    neighbours[i].append((j, R))
    neighbours[j].append((i, R))

def h_flow(p_i, p_j, R_ij):
    return np.sign(p_i - p_j) * np.sqrt(abs(p_i - p_j) / R_ij)

def h_injection(i, p, neighbours):
    total = 0.0
    for k, R_ik in neighbours[i]:
        total += h_flow(p[i], p[k], R_ik)
    return -total
N = 7
p = np.array([1.00, 0.99, 0.97, 0.96, 0.98, 0.94, 0.92])
print(sum(h_injection(i, p, neighbours) for i in range(N)))


5.551115123125783e-17
